# Colab Workshop: Ollama + RAG + Advanced Retrieval

This notebook is designed for **Google Colab** and has three sections:
1. Setup Ollama
2. Intro to RAG
3. Advanced retrieval

## How to Enable the Free GPU

Ask your students to do the following as their very first step:

- Open the Menu: Go to Runtime in the top navigation bar.
- Change Type: Select Change runtime type.
- Select GPU: Under the Hardware accelerator dropdown, choose T4 GPU (this is the standard free tier GPU).
- Save: Click Save. The notebook will briefly reconnect to a new virtual machine equipped with the GPU.

## How To Use This Notebook

Run the notebook from top to bottom in a fresh Colab runtime.

Expected timing on free Colab:
- Setup and model pull: 5 to 20 minutes
- Index build and first query: 2 to 8 minutes

If a runtime disconnects, restart and rerun all cells in order.

## 1) Setup Ollama

### What This Section Does

In this section you will:
- Install required Python packages
- Install and start the Ollama server in Colab
- Pull one local LLM and one embedding model
- Run a quick smoke test to confirm local inference works

After this section, you should be able to run LLM calls without any external API key.

In [ ]:
# Install Python dependencies for LlamaIndex + Ollama integrations
%pip install -q jedi llama-index llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
# Install required system dependency and Ollama binary in Colab
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama server (safe to rerun)
import os
import time
import subprocess

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

check = subprocess.run(["bash", "-lc", "pgrep -f 'ollama serve'"], capture_output=True, text=True)
if check.returncode != 0:
    _ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(4)
    print("Started Ollama server.")
else:
    print("Ollama server is already running.")

!ollama list

In [ ]:
# Pull lightweight models for Colab
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text

In [ ]:
# Quick smoke test
!ollama run llama3.2:3b "Reply with: Ollama is ready in Colab."

### Setup Checkpoint

If the smoke test responded correctly, your local model is ready.

If not:
- Re-run the server start cell
- Re-run the model pull cell
- Confirm `ollama list` shows `llama3.2:3b` and `nomic-embed-text`

## 2) Intro to RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: use those chunks to produce an answer

This reduces hallucination by grounding answers in retrieved source text.

In [ ]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="llama3.2:3b", request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("LLM and embedding model configured.")

In [ ]:
# Download sample text for indexing
import os
import urllib.request

os.makedirs("data", exist_ok=True)
essay_path = "data/paul_graham_essay.txt"

if not os.path.exists(essay_path):
    url = (
        "https://raw.githubusercontent.com/run-llama/llama_index/"
        "main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt"
    )
    urllib.request.urlretrieve(url, essay_path)
    print(f"Downloaded essay to {essay_path}")
else:
    print(f"Essay already exists at {essay_path}")

In [ ]:
# Build a basic RAG index
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print(f"Loaded {len(documents)} document(s) and built index.")

In [ ]:
# Intro RAG query
response = query_engine.query("What did Paul Graham work on before Y Combinator?")
print(response)

## 3) Advanced Retrieval

### Why Advanced Retrieval Matters

A basic retriever may return partially relevant chunks.
Reranking adds an extra scoring step to keep the most relevant context before final answer generation.

In practice, this often improves factual precision and relevance.

In [ ]:
# Baseline retrieval
query = "What did Paul Graham work on at Viaweb and what did he learn from it?"
baseline_engine = index.as_query_engine(similarity_top_k=6)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

### Baseline First

This response uses only vector similarity retrieval.
Use it as a reference point before applying reranking.

In [ ]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

reranker = LLMRerank(choice_batch_size=4, top_n=3)
rerank_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)

### Add Reranking

This step retrieves a wider candidate set, then uses the LLM to keep only the top chunks before answering.
Compare this response with the baseline to see how retrieval quality changes.

In [ ]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

In [ ]:
# Try your own question
your_question = input("Ask a question about the essay: ")
print(rerank_engine.query(your_question))

### Suggested Exercises

Try these quick experiments:
- Change `similarity_top_k` from 4 to 8 in intro RAG
- Change reranker `top_n` from 3 to 2 or 5
- Ask multi-part questions and observe whether reranking helps